In [2]:
import pandas as pd
from pathlib import Path
import re

In [3]:
DATA_DIR = Path("./data/labeled")
OUTPUT_FILE = DATA_DIR / "Chile_all1_clean.csv"
PART_FILE_PATTERN = "Chile_all1_clean_labeled_*.csv"

In [4]:
def natural_part_key(path: Path) -> int:
    match = re.search(r"part(\d+)", path.stem)
    if not match:
        return 10**9
    return int(match.group(1))

In [5]:
def read_tripadvisor_data(path: str | Path) -> pd.DataFrame:
    data_path = Path(path)
    files = sorted(data_path.glob(PART_FILE_PATTERN), key=natural_part_key)
    if not files:
        raise FileNotFoundError(
            f"No se encontraron archivos con patron {PART_FILE_PATTERN} en {data_path}"
        )

    dataframes = [pd.read_csv(file_path, sep=";", encoding="utf-8") for file_path in files]
    return pd.concat(dataframes, ignore_index=True)

In [6]:
df_raw = read_tripadvisor_data(DATA_DIR)

In [7]:
df_labeled = df_raw.copy()
df_labeled.columns

Index(['review_text', 'title', 'language', 'language_detected_context',
       'language_detected_full', 'language_majority_lang',
       'language_majority_percentage', 'review_text_preprocessed',
       'razonamiento_corto', 'etiqueta', 'estado_etiquetado'],
      dtype='str')

In [9]:
# ═══════════════════════════════════════════════════════════════════
#  ModernBERT — Fine-tuning v3
#  40K comentarios: 38141 NON-SUGGESTION / 2588 SUGGESTION
# ═══════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch import optim
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split

# ───────────────────────────────────────────────────────────────────
#  0. CONFIGURACIÓN
# ───────────────────────────────────────────────────────────────────
MODEL_ID   = "answerdotai/ModernBERT-base"
MAX_LEN    = 512
BATCH_SIZE = 32       # más grande — tenemos más datos y es más estable
EPOCHS     = 8
LR         = 1e-5     # más bajo que antes — descongelamos más capas
SEED       = 42
GRAD_ACCUM = 2        # acumulación de gradientes → batch efectivo = 64

LABEL2ID = {"NON-SUGGESTION": 0, "SUGGESTION": 1}
ID2LABEL = {0: "NON-SUGGESTION", 1: "SUGGESTION"}

torch.manual_seed(SEED)
device = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Dispositivo: {device}")


# ───────────────────────────────────────────────────────────────────
#  1. DATOS Y SPLIT
# ───────────────────────────────────────────────────────────────────
df = df_labeled[df_labeled["etiqueta"].isin(LABEL2ID)].copy()
df["label_id"] = df["etiqueta"].map(LABEL2ID)

print(f"Total: {len(df)}")
print(df["etiqueta"].value_counts())

df_train, df_temp = train_test_split(
    df, test_size=0.20, stratify=df["label_id"], random_state=SEED
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, stratify=df_temp["label_id"], random_state=SEED
)

print(f"\nTrain : {len(df_train)} | {df_train['etiqueta'].value_counts().to_dict()}")
print(f"Val   : {len(df_val)}   | {df_val['etiqueta'].value_counts().to_dict()}")
print(f"Test  : {len(df_test)}  | {df_test['etiqueta'].value_counts().to_dict()}")


# ───────────────────────────────────────────────────────────────────
#  2. DATASET — sin WeightedRandomSampler
#     El desbalance lo maneja Focal Loss (más adelante)
# ───────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

class SuggestionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = [str(t) for t in texts]
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_loader = DataLoader(
    SuggestionDataset(df_train["review_text"].tolist(),
                      df_train["label_id"].tolist()),
    batch_size=BATCH_SIZE, shuffle=True      # shuffle normal, sin sampler
)
val_loader = DataLoader(
    SuggestionDataset(df_val["review_text"].tolist(),
                      df_val["label_id"].tolist()),
    batch_size=64, shuffle=False
)
test_loader = DataLoader(
    SuggestionDataset(df_test["review_text"].tolist(),
                      df_test["label_id"].tolist()),
    batch_size=64, shuffle=False
)


# ───────────────────────────────────────────────────────────────────
#  3. MODELO — backbone + cabeza custom con dropout
#     Descongelamos las últimas 4 capas transformer.
#     Si quieres experimentar con "solo capas densas" cambia
#     N_UNFREEZE = 0, pero los resultados serán peores.
# ───────────────────────────────────────────────────────────────────
N_UNFREEZE = 4   # número de capas transformer a descongelar

backbone = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# Congelar todo
for param in backbone.parameters():
    param.requires_grad = False

# Descongelar últimas N_UNFREEZE capas
for layer in backbone.model.layers[-N_UNFREEZE:]:
    for param in layer.parameters():
        param.requires_grad = True

# Reemplazar la cabeza original por una más expresiva con dropout
# hidden_size de ModernBERT-base = 768
hidden_size = backbone.config.hidden_size

backbone.classifier = nn.Sequential(
    nn.Linear(hidden_size, 512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512, 128),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(128, 2),          # salida: [logit_nonsugg, logit_sugg]
)

# La cabeza siempre es entrenable
for param in backbone.classifier.parameters():
    param.requires_grad = True

model = backbone.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nParámetros entrenables: {trainable:,} / {total:,} "
      f"({100*trainable/total:.1f}%)")


# ───────────────────────────────────────────────────────────────────
#  4. FOCAL LOSS — diseñada para desbalance de clases
#
#  Ventaja sobre CrossEntropy + weight:
#  - CE ponderada penaliza todos los errores de la clase minoritaria
#    por igual, lo que empuja al modelo a sobredisparar.
#  - Focal Loss penaliza más los ejemplos DIFÍCILES (baja confianza)
#    y menos los fáciles (alta confianza), independiente de la clase.
#    Resultado: precisión más alta sin sacrificar recall.
#
#  gamma=2 es el valor estándar (paper original de RetinaNet).
#  alpha=0.85 = peso para la clase positiva (SUGGESTION).
#    → calculado como n_nonsugg / (n_nonsugg + n_sugg)
# ───────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.85, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        ce_loss = nn.functional.cross_entropy(
            logits, targets, reduction="none"
        )
        pt      = torch.exp(-ce_loss)                  # probabilidad predicha
        alpha_t = torch.where(
            targets == 1,
            torch.tensor(self.alpha, device=logits.device, dtype=torch.float32),
            torch.tensor(1 - self.alpha, device=logits.device, dtype=torch.float32),
        )
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

n_nonsugg = (df_train["label_id"] == 0).sum()
n_sugg    = (df_train["label_id"] == 1).sum()
alpha     = n_nonsugg / (n_nonsugg + n_sugg)  # ≈ 0.936

criterion = FocalLoss(alpha=alpha, gamma=2.0)
print(f"Focal Loss — alpha: {alpha:.3f}, gamma: 2.0")


# ───────────────────────────────────────────────────────────────────
#  5. OPTIMIZADOR CON LEARNING RATES DIFERENCIALES
#     Las capas más profundas aprenden más despacio que la cabeza
#     (técnica estándar en fine-tuning de transformers)
# ───────────────────────────────────────────────────────────────────
optimizer = optim.AdamW([
    {"params": backbone.model.layers[-N_UNFREEZE:].parameters(),
     "lr": LR},                  # capas transformer: LR base
    {"params": backbone.classifier.parameters(),
     "lr": LR * 10},             # cabeza: 10× más rápida
], weight_decay=0.01)

total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = total_steps // 10

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)


# ───────────────────────────────────────────────────────────────────
#  6. ENTRENAMIENTO
# ───────────────────────────────────────────────────────────────────
def evaluate(loader):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["label"].to(device)

            logits = model(input_ids=ids, attention_mask=mask).logits
            probs  = torch.softmax(logits, dim=1)
            preds  = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(lbls.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    f1   = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)
    prec = precision_score(all_targets, all_preds, pos_label=1, zero_division=0)
    rec  = recall_score(all_targets, all_preds, pos_label=1, zero_division=0)
    return all_preds, all_targets, np.array(all_probs), \
           {"f1": f1, "precision": prec, "recall": rec}


best_val_f1  = 0.0
patience     = 3
no_improve   = 0
best_weights = None

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids     = batch["input_ids"].to(device)
        mask    = batch["attention_mask"].to(device)
        targets = batch["label"].to(device)

        logits = model(input_ids=ids, attention_mask=mask).logits
        loss   = criterion(logits, targets) / GRAD_ACCUM  # normalizar
        loss.backward()

        # Actualizar pesos cada GRAD_ACCUM pasos
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM

    avg_loss = total_loss / len(train_loader)
    _, _, _, val_m = evaluate(val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | "
        f"Val F1: {val_m['f1']:.4f} | "
        f"P: {val_m['precision']:.4f} | R: {val_m['recall']:.4f}"
    )

    if val_m["f1"] > best_val_f1:
        best_val_f1  = val_m["f1"]
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve   = 0
        print(f"  ✓ Mejor Val F1: {best_val_f1:.4f} — guardado")
    else:
        no_improve += 1
        print(f"  Sin mejora ({no_improve}/{patience})")
        if no_improve >= patience:
            print("  Early stopping.")
            break

model.load_state_dict(best_weights)
print(f"\nPesos restaurados — mejor Val F1: {best_val_f1:.4f}")


# ───────────────────────────────────────────────────────────────────
#  7. EVALUACIÓN FINAL + THRESHOLD ÓPTIMO
# ───────────────────────────────────────────────────────────────────
preds, targets, probs, test_m = evaluate(test_loader)
prob_sugg = probs[:, 1]

print("\n" + "═" * 55)
print("  EVALUACIÓN FINAL — TEST SET (threshold=0.5)")
print("═" * 55)
print(f"  F1: {test_m['f1']:.4f} | P: {test_m['precision']:.4f} "
      f"| R: {test_m['recall']:.4f}")

cm = confusion_matrix(targets, preds)
tn, fp, fn, tp = cm.ravel()
print(f"\n  Matriz de Confusión:")
print(f"  {'':22s}  Pred NON-SUGG  Pred SUGG")
print(f"  {'Real NON-SUGG':22s}  {tn:13d}  {fp:9d}")
print(f"  {'Real SUGG':22s}  {fn:13d}  {tp:9d}")
print(f"\n{classification_report(targets, preds, target_names=['NON-SUGGESTION','SUGGESTION'], zero_division=0)}")

# Buscar threshold óptimo automáticamente
print("  Análisis de threshold:")
print(f"  {'Thresh':>8}  {'P':>8}  {'R':>8}  {'F1':>8}  {'SUGG':>8}")
best_t, best_f = 0.5, 0.0
for t in np.arange(0.1, 0.95, 0.05):
    pt = (prob_sugg >= t).astype(int)
    p  = precision_score(targets, pt, pos_label=1, zero_division=0)
    r  = recall_score(targets, pt, pos_label=1, zero_division=0)
    f  = f1_score(targets, pt, pos_label=1, zero_division=0)
    n  = pt.sum()
    if f > best_f:
        best_f, best_t = f, t
    mark = " ← ÓPTIMO" if t == best_t else \
           " ← default" if abs(t - 0.5) < 0.01 else ""
    print(f"  {t:>8.2f}  {p:>8.4f}  {r:>8.4f}  {f:>8.4f}  {n:>8d}{mark}")

print(f"\n  Threshold óptimo: {best_t:.2f} (F1={best_f:.4f})")


# ───────────────────────────────────────────────────────────────────
#  8. INFERENCIA INDIVIDUAL
# ───────────────────────────────────────────────────────────────────
def classify(text: str, threshold: float = None) -> dict:
    """threshold=None usa el óptimo encontrado en el test set."""
    if threshold is None:
        threshold = best_t

    model.eval()
    enc = tokenizer(
        str(text), max_length=MAX_LEN, padding="max_length",
        truncation=True, return_tensors="pt",
    )
    with torch.no_grad():
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device)
        ).logits
        p = torch.softmax(logits, dim=1)[0].cpu().numpy()

    return {
        "label": "SUGGESTION" if p[1] >= threshold else "NON-SUGGESTION",
        "probabilities": {
            "NON-SUGGESTION": round(float(p[0]), 4),
            "SUGGESTION":     round(float(p[1]), 4),
        },
        "threshold": round(threshold, 2),
    }


ejemplos = [
    ("The bed was hard and the room smelled bad.",
     "NON-SUGGESTION esperado"),
    ("They should add an English translation option for visitors.",
     "SUGGESTION esperado"),
    ("Amazing tour with Pedro, I highly recommend him!",
     "NON-SUGGESTION esperado"),
    ("It would be great if they included a map at the entrance.",
     "SUGGESTION esperado"),
    ("Beautiful place, definitely visit in summer.",
     "NON-SUGGESTION esperado"),
]

print(f"\n  Inferencia individual (threshold óptimo={best_t:.2f}):")
for texto, esperado in ejemplos:
    r   = classify(texto)
    bar = "█" * int(r["probabilities"]["SUGGESTION"] * 20)
    print(f"\n  [{esperado}]")
    print(f"  {texto[:80]}")
    print(f"  → {r['label']} | p(NON)={r['probabilities']['NON-SUGGESTION']:.4f} "
          f"p(SUGG)={r['probabilities']['SUGGESTION']:.4f}  [{bar:<20}]")

Dispositivo: mps
Total: 40729
etiqueta
NON-SUGGESTION    38141
SUGGESTION         2588
Name: count, dtype: int64

Train : 32583 | {'NON-SUGGESTION': 30513, 'SUGGESTION': 2070}
Val   : 4073   | {'NON-SUGGESTION': 3814, 'SUGGESTION': 259}
Test  : 4073  | {'NON-SUGGESTION': 3814, 'SUGGESTION': 259}


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Parámetros entrenables: 20,519,810 / 150,064,514 (13.7%)
Focal Loss — alpha: 0.936, gamma: 2.0
Epoch 1/8 | Loss: 0.0171 | Val F1: 0.2300 | P: 0.1327 | R: 0.8610
  ✓ Mejor Val F1: 0.2300 — guardado
Epoch 2/8 | Loss: 0.0139 | Val F1: 0.3465 | P: 0.2215 | R: 0.7954
  ✓ Mejor Val F1: 0.3465 — guardado
Epoch 3/8 | Loss: 0.0124 | Val F1: 0.3429 | P: 0.2158 | R: 0.8340
  Sin mejora (1/3)


KeyboardInterrupt: 